<a href="https://colab.research.google.com/github/equiphysics/education/blob/main/Final_Project_Reel_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project Reel Generator — Horses & Music HRV Study

This notebook creates portrait-format (9:16) Instagram Reels showing horse video synchronized with scrolling HR, RMSSD, loudness, and spectral centroid graphs.

**What you need:**
1. A video file (`.mov` or `.mp4`) of the horse session
2. A Polar HR data file (`.txt`) from the Polar Equine monitor
3. The time lag (in seconds) between when the video starts and when the HR recording starts

**How to use:**
1. Run all cells in order
2. Upload your video and HR files when prompted
3. Enter the session metadata (horse name, music type, time lag)
4. The reel will be generated and available for download

In [ ]:
# @title Cell 1: Setup & Imports
import subprocess, sys, os, wave, json, tempfile, shutil, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Check for ffmpeg
try:
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print('ffmpeg available:', result.stdout.split('\n')[0])
except FileNotFoundError:
    print('ERROR: ffmpeg not found.')

# Check for Google Colab
IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print('Running in Google Colab - file upload/download enabled')
except ImportError:
    print('Not running in Colab - will use local file paths')

print('Setup complete!')

In [ ]:
# @title Cell 2: Video Configuration
WIDTH, HEIGHT = 1080, 1920       # Instagram Reel dimensions (9:16 portrait)
VIDEO_HEIGHT = 740                # Top portion for video
GRAPH_HEIGHT = HEIGHT - VIDEO_HEIGHT  # Bottom portion for graphs
FPS = 30
AXIS_W = 130                     # Fixed left axis panel width
DATA_W = WIDTH - AXIS_W          # Scrollable data area (950 px)

# Colors (dark theme)
BG_COLOR = '#1a1a2e'
GRAPH_BG = '#16213e'
GRID_COLOR = '#2a3a5e'
HR_COLOR = '#ff69b4'              # Pink for heart rate
RMSSD_COLOR = '#00e5ff'           # Bright cyan for RMSSD
LOUD_COLOR = '#e9b044'            # Gold for loudness
CENT_COLOR = '#53d8a8'            # Teal for spectral centroid
TEXT_COLOR = '#e0e0e0'
CURSOR_COLOR = '#FF6B35'          # Orange cursor line

PX_PER_SEC = 15                   # Pixels per second in data strip
TITLE_H = 50                      # Title bar height
PLOT_AREA_H = GRAPH_HEIGHT - TITLE_H
YLABEL_SIZE = 14
YTICK_SIZE = 12
TITLE_SIZE = 18

print(f'Configuration loaded: {WIDTH}x{HEIGHT} at {FPS} fps')

In [ ]:
# @title Cell 3: Upload Files & Enter Session Info
# Fill in the form on the right, then run this cell.
# A file picker or path prompt will appear below depending on your upload method.

HORSE_NAME = ""         # @param {type:"string"}
MUSIC_LABEL = ""        # @param {type:"string"}
TIME_LAG = 0.0          # @param {type:"number"}
UPLOAD_METHOD = "Upload from computer"  # @param ["Upload from computer", "Google Drive"]

# ── Nothing to edit below this line ──────────────────────

import os
from IPython.display import display, HTML

def _bar(step, total, msg):
    pct = int(step / total * 100)
    filled = "\u2588" * (pct // 5) + "\u2591" * (20 - pct // 5)
    print(f"  [{filled}] {pct}% \u2014 {msg}")

def _stop(html_msg):
    """Show a styled message and stop the cell without a traceback."""
    display(HTML(html_msg))
    class StopExecution(Exception):
        def _render_traceback_(self): return []
    raise StopExecution()

print("=" * 55)
print("  \U0001f3ac  Upload Files & Enter Session Info")
print("=" * 55)

# --- Validate session info ---
missing = []
if not HORSE_NAME.strip():
    missing.append("<b>HORSE_NAME</b> \u2014 e.g., Jett, Perseo, Dude, Duque")
if not MUSIC_LABEL.strip():
    missing.append("<b>MUSIC_LABEL</b> \u2014 e.g., Opera, Rock Lobster, Silent")

if missing:
    items = "".join(f"<li>{m}</li>" for m in missing)
    _stop(
        "<div style='padding:12px;background:#614a00;color:#ffe28a;"
        "border:1px solid #ffc107;border-radius:6px;margin:8px 0'>"
        "<b>\u270b Almost there!</b> Fill in the form on the right:"
        f"<ul style='margin:6px 0'>{items}</ul>"
        "Then re-run this cell.</div>"
    )

_bar(1, 4, "Session info OK")

# --- Load files ---
VIDEO_PATH = None
HR_PATH = None

if IN_COLAB:
    if UPLOAD_METHOD == "Google Drive":
        _bar(2, 4, "Mounting Google Drive...")
        from google.colab import drive
        drive.mount("/content/drive")
        print()
        print("    Enter the full path to each file on your Google Drive.")
        print("    Tip: right-click a file in the Colab file browser \u2192 Copy path")
        print("    Example: /content/drive/My Drive/Horses/Jett-Opera.mov")
        print()
        VIDEO_PATH = input("    \U0001f3ac Video file path: ").strip()
        HR_PATH = input("    \u2764\ufe0f HR file path:    ").strip()

        if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
            _stop(
                "<div style='padding:12px;background:#5c1a1a;color:#ffb3b3;"
                "border:1px solid #ff4444;border-radius:6px;margin:8px 0'>"
                f"\u274c <b>Video not found:</b> <code>{VIDEO_PATH}</code><br>"
                "Check the path and re-run this cell.</div>"
            )
        if not HR_PATH or not os.path.exists(HR_PATH):
            _stop(
                "<div style='padding:12px;background:#5c1a1a;color:#ffb3b3;"
                "border:1px solid #ff4444;border-radius:6px;margin:8px 0'>"
                f"\u274c <b>HR file not found:</b> <code>{HR_PATH}</code><br>"
                "Check the path and re-run this cell.</div>"
            )
        v_mb = os.path.getsize(VIDEO_PATH) / 1e6
        h_kb = os.path.getsize(HR_PATH) / 1e3
        _bar(3, 4, "Files found on Drive")
        print(f"    \u2705 Video: {os.path.basename(VIDEO_PATH)} ({v_mb:.1f} MB)")
        print(f"    \u2705 HR:    {os.path.basename(HR_PATH)} ({h_kb:.0f} KB)")

    else:  # Upload from computer
        _bar(2, 4, "Waiting for video upload...")
        print()
        print("    \u2b06\ufe0f Click 'Choose Files' below to select your VIDEO (.mov or .mp4)")
        print()
        uploaded_video = colab_files.upload()
        if not uploaded_video:
            _stop(
                "<div style='padding:12px;background:#5c1a1a;color:#ffb3b3;"
                "border:1px solid #ff4444;border-radius:6px;margin:8px 0'>"
                "\u274c No video uploaded. Re-run this cell to try again.</div>"
            )
        video_filename = list(uploaded_video.keys())[0]
        VIDEO_PATH = os.path.join("/content", video_filename)
        with open(VIDEO_PATH, "wb") as f:
            f.write(uploaded_video[video_filename])
        v_mb = len(uploaded_video[video_filename]) / 1e6
        print(f"    \u2705 Video saved: {video_filename} ({v_mb:.1f} MB)")
        print()

        _bar(3, 4, "Waiting for HR file upload...")
        print()
        print("    \u2b06\ufe0f Click 'Choose Files' below to select your POLAR HR file (.txt)")
        print()
        uploaded_hr = colab_files.upload()
        if not uploaded_hr:
            _stop(
                "<div style='padding:12px;background:#5c1a1a;color:#ffb3b3;"
                "border:1px solid #ff4444;border-radius:6px;margin:8px 0'>"
                "\u274c No HR file uploaded. Re-run this cell to try again.</div>"
            )
        hr_filename = list(uploaded_hr.keys())[0]
        HR_PATH = os.path.join("/content", hr_filename)
        with open(HR_PATH, "wb") as f:
            f.write(uploaded_hr[hr_filename])
        h_kb = len(uploaded_hr[hr_filename]) / 1e3
        print(f"    \u2705 HR saved: {hr_filename} ({h_kb:.0f} KB)")

else:
    # Not in Colab
    VIDEO_PATH = input("Enter the full path to your video file: ").strip()
    HR_PATH = input("Enter the full path to your Polar HR file: ").strip()

print()
_bar(4, 4, "Ready!")

display(HTML(
    "<div style='padding:12px;background:#1a3d1a;color:#b3ffb3;"
    "border:1px solid #44ff44;border-radius:6px;margin:8px 0'>"
    "<b>\u2705 All set!</b><br>"
    f"\u2022 Horse: <b>{HORSE_NAME}</b><br>"
    f"\u2022 Music: <b>{MUSIC_LABEL}</b><br>"
    f"\u2022 Time lag: <b>{TIME_LAG}s</b><br>"
    f"\u2022 Video: <b>{os.path.basename(VIDEO_PATH)}</b><br>"
    f"\u2022 HR file: <b>{os.path.basename(HR_PATH)}</b><br><br>"
    "Run <b>Cell 4</b> (processing functions) then <b>Cell 5</b> (generate reel).</div>"
))
print("=" * 55)


In [ ]:
# @title Cell 4: Processing Functions (run this cell, no output expected)

def read_polar_hr(path):
    """Read Polar Equine HR monitor data file."""
    with open(path, 'rb') as f:
        raw = f.read().replace(b'\x00', b'')
    lines = raw.decode('utf-8', errors='replace').splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        s = line.strip()
        if s == '' or s.startswith('#'):
            continue
        header_idx = i
        break
    if header_idx is None:
        raise ValueError('Could not find data header in HR file')
    headers = [h.strip() for h in lines[header_idx].split(',')]
    rows = []
    for line in lines[header_idx+1:]:
        s = line.strip()
        if s == '' or s.startswith('#'):
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= len(headers):
            rows.append(dict(zip(headers, parts[:len(headers)])))
    if not rows:
        raise ValueError('No data rows found in HR file')
    df = pd.DataFrame(rows)
    hr_col = next((c for c in df.columns if c.strip().upper() == 'HR'), None)
    rr_col = next((c for c in df.columns if c.strip().upper() == 'RR'), None)
    ms_col = next((c for c in df.columns if c.strip().upper() == 'MS'), None)
    hr = pd.to_numeric(df[hr_col], errors='coerce').values.astype(float) if hr_col else np.zeros(len(df))
    rr = pd.to_numeric(df[rr_col], errors='coerce').values.astype(float) if rr_col else np.zeros(len(df))
    ts = (pd.to_numeric(df[ms_col], errors='coerce').values.astype(float) / 1000.0) if ms_col else np.cumsum(rr) / 1000.0
    return ts, hr, rr


def compute_rolling_metrics(ts, hr, rr, window_s=30, hop_s=1):
    """Compute rolling HR and RMSSD from R-R intervals."""
    rr = rr.astype(float)
    valid = (rr > 200) & (rr < 3000)
    t_max = ts[-1] if len(ts) > 0 else 0
    times, hr_r, rmssd_r = [], [], []
    for tc in np.arange(window_s/2, t_max - window_s/2, hop_s):
        mask = (ts >= tc - window_s/2) & (ts < tc + window_s/2) & valid
        seg = rr[mask]
        if len(seg) < 5:
            continue
        times.append(tc)
        hr_r.append(60000.0 / np.mean(seg))
        d = np.diff(seg)
        rmssd_r.append(np.sqrt(np.mean(d**2)) if len(d) > 0 else np.nan)
    return np.array(times), np.array(hr_r), np.array(rmssd_r)


def extract_audio_features(video_path, sr=11025, win_sec=0.5, hop_sec=0.1):
    """Extract loudness (dB) and spectral centroid from video audio."""
    wav_path = '/tmp/reel_audio.wav'
    subprocess.run(['ffmpeg', '-y', '-i', str(video_path), '-vn', '-acodec', 'pcm_s16le',
                    '-ar', str(sr), '-ac', '1', wav_path], capture_output=True, timeout=120)
    w = wave.open(wav_path, 'rb')
    raw = w.readframes(w.getnframes())
    actual_sr = w.getframerate()
    w.close()
    samples = np.frombuffer(raw, dtype=np.int16).astype(float) / 32768.0
    win_n = int(win_sec * actual_sr)
    hop_n = int(hop_sec * actual_sr)
    times, loudness, centroids = [], [], []
    for start in range(0, len(samples) - win_n, hop_n):
        frame = samples[start:start+win_n]
        t = (start + win_n/2) / actual_sr
        rms = np.sqrt(np.mean(frame**2))
        windowed = frame * np.hanning(win_n)
        spec = np.abs(np.fft.rfft(windowed))
        freqs = np.fft.rfftfreq(win_n, 1.0/actual_sr)
        centroid = np.sum(freqs * spec) / np.sum(spec) if np.sum(spec) > 0 else 0.0
        times.append(t)
        loudness.append(rms)
        centroids.append(centroid)
    os.remove(wav_path)
    return np.array(times), np.array(loudness), np.array(centroids)


def get_video_duration(video_path):
    """Get video duration in seconds using ffprobe."""
    result = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
                            '-show_format', str(video_path)], capture_output=True, text=True)
    return float(json.loads(result.stdout)['format']['duration'])


def compute_data_ranges(hr_vals, rmssd_vals, loud_db, cent_vals):
    """Pre-compute y-axis ranges so axes and data images match exactly."""
    ranges = []
    for vals, label in [(hr_vals, 'HR'), (rmssd_vals, 'RMSSD'),
                        (loud_db, 'Loud'), (cent_vals, 'Cent')]:
        if len(vals) > 0:
            ymin, ymax = np.nanmin(vals), np.nanmax(vals)
            pad = max((ymax - ymin) * 0.15, 1)
            is_db = (label == 'Loud')
            lo = ymin - pad if is_db else max(0, ymin - pad)
            hi = ymax + pad
        else:
            lo, hi = 0, 1
        ranges.append((lo, hi))
    return ranges


def render_axis_image(ranges, out_path):
    """Render a fixed-width image with y-axis labels and ticks."""
    dpi = 100
    fig_w = AXIS_W / dpi
    fig_h = PLOT_AREA_H / dpi
    fig, axes = plt.subplots(4, 1, figsize=(fig_w, fig_h), facecolor=BG_COLOR)
    labels = ['HR\n(bpm)', 'RMSSD\n(ms)', 'Loud\n(dB)', 'Centroid\n(kHz)']
    colors = [HR_COLOR, RMSSD_COLOR, LOUD_COLOR, CENT_COLOR]
    for i, ax in enumerate(axes):
        ax.set_facecolor(BG_COLOR)
        ax.set_xlim(0, 1)
        ax.set_xticks([])
        if i == 3:
            lo_khz, hi_khz = ranges[i][0] / 1000.0, ranges[i][1] / 1000.0
            ax.set_ylim(lo_khz, hi_khz)
        else:
            ax.set_ylim(ranges[i])
        ax.yaxis.tick_right()
        if i == 0:
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=True))
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))
        elif i == 3:
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
        else:
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=False))
        ax.tick_params(axis='y', colors=TEXT_COLOR, labelsize=YTICK_SIZE,
                       right=False, left=False,
                       labelright=True, labelleft=False,
                       length=0, pad=2)
        ax.text(0.05, 0.5, labels[i], transform=ax.transAxes,
                fontsize=YLABEL_SIZE, fontweight='bold', color=colors[i],
                ha='left', va='center', linespacing=1.2)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.spines['right'].set_visible(True)
        ax.spines['right'].set_color(GRID_COLOR)
    fig.subplots_adjust(left=0.02, right=0.58, top=0.98, bottom=0.03, hspace=0.35)
    fig.savefig(out_path, dpi=dpi, facecolor=BG_COLOR)
    plt.close(fig)


def render_data_image(t_max, hr_t, hr_vals, rmssd_t, rmssd_vals,
                      loud_t, loud_db, cent_t, cent_vals, ranges, out_path):
    """Render the wide scrollable data strip."""
    total_w_px = int(t_max * PX_PER_SEC)
    total_w_px = max(total_w_px, DATA_W)
    dpi = 100
    fig_w = total_w_px / dpi
    fig_h = PLOT_AREA_H / dpi
    fig, axes = plt.subplots(4, 1, figsize=(fig_w, fig_h), facecolor=BG_COLOR)
    cent_khz = cent_vals / 1000.0
    plot_data = [
        (axes[0], hr_t, hr_vals, HR_COLOR, True),
        (axes[1], rmssd_t, rmssd_vals, RMSSD_COLOR, False),
        (axes[2], loud_t, loud_db, LOUD_COLOR, True),
        (axes[3], cent_t, cent_khz, CENT_COLOR, True),
    ]
    time_ticks = np.arange(0, t_max + 1, 30)
    for i, (ax, t_data, y_data, color, fill) in enumerate(plot_data):
        ax.set_facecolor(GRAPH_BG)
        if len(t_data) > 0:
            ax.plot(t_data, y_data, color=color, linewidth=2.5, alpha=0.9)
            if fill:
                ax.fill_between(t_data, y_data, alpha=0.12, color=color)
        ax.set_xlim(0, t_max)
        if i == 3:
            ax.set_ylim(ranges[i][0] / 1000.0, ranges[i][1] / 1000.0)
        else:
            ax.set_ylim(ranges[i])
        if i == 0:
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=True))
        else:
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=False))
        ax.tick_params(axis='y', which='both', colors=TEXT_COLOR,
                       left=True, right=False,
                       labelleft=False, labelright=False,
                       length=6, width=1.5, direction='in')
        ax.set_ylabel('')
        ax.set_xticks(time_ticks)
        ax.tick_params(axis='x', colors=TEXT_COLOR, labelsize=YTICK_SIZE)
        ax.grid(True, color=GRID_COLOR, alpha=0.4, linewidth=0.5)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['bottom'].set_color(GRID_COLOR)
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color(GRID_COLOR)
    def fmt_time(x, pos):
        m, s = divmod(int(x), 60)
        return f'{m}:{s:02d}'
    axes[3].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_time))
    for ax in axes[:3]:
        ax.set_xticklabels([])
    fig.subplots_adjust(left=0.0, right=1.0, top=0.98, bottom=0.04, hspace=0.35)
    fig.savefig(out_path, dpi=dpi, facecolor=BG_COLOR)
    plt.close(fig)
    return total_w_px


def render_title_bar(horse_name, music_label, summary, out_path):
    """Render a title bar image."""
    dpi = 100
    fig, ax = plt.subplots(figsize=(WIDTH/dpi, TITLE_H/dpi), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    ax.axis('off')
    ax.text(0.5, 0.5, f'{horse_name}  \u2022  {music_label}  \u2022  {summary}',
            fontsize=TITLE_SIZE, color='white', fontweight='bold',
            ha='center', va='center', transform=ax.transAxes)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(out_path, dpi=dpi, facecolor=BG_COLOR)
    plt.close(fig)

print('All processing functions loaded!')

In [ ]:
# @title Cell 5: Generate the Reel
# This cell processes your data and creates the video.
# It may take several minutes depending on session length.

from IPython.display import display, HTML
import time as _time

start_time = _time.time()

# --- Visual progress bar ---
_steps = [
    "Reading video duration",
    "Processing heart rate data",
    "Extracting audio features",
    "Rendering axis panel",
    "Rendering data strip",
    "Rendering title bar & preparing video",
    "Creating scrolling graph video",
    "Composing final video"
]
_total_steps = len(_steps)

def _progress(step_num, extra=""):
    pct = int(step_num / _total_steps * 100)
    label = _steps[step_num - 1] if step_num <= len(_steps) else "Done"
    bar_html = (
        f"<div style='background:#2a2a4a;border-radius:8px;padding:2px;margin:4px 0'>"
        f"<div style='background:linear-gradient(90deg,#ff69b4,#00e5ff);width:{pct}%;"
        f"padding:6px 12px;border-radius:6px;color:white;font-weight:bold;"
        f"font-size:13px;text-align:right;min-width:60px;transition:width 0.3s'>"
        f"{pct}%</div></div>"
        f"<div style='color:#aaa;font-size:12px;margin:2px 0 8px 4px'>"
        f"[{step_num}/{_total_steps}] {label}"
        f"{'  —  ' + extra if extra else ''}</div>"
    )
    if not hasattr(_progress, '_handle'):
        _progress._handle = display(HTML(bar_html), display_id=True)
    else:
        _progress._handle.update(HTML(bar_html))

print("=" * 60)
print(f"Generating reel: {HORSE_NAME} - {MUSIC_LABEL}")
print("=" * 60)

# Step 1: Video duration
_progress(1)
duration = get_video_duration(VIDEO_PATH)
_progress(1, f"{duration:.0f}s ({duration/60:.1f} min)")

# Step 2: HR data
_progress(2)
ts_raw, hr_raw, rr_raw = read_polar_hr(HR_PATH)
ts_aligned = ts_raw + TIME_LAG
hr_t, hr_rolling, rmssd_rolling = compute_rolling_metrics(ts_aligned, hr_raw, rr_raw, window_s=30, hop_s=1)
valid_rr = rr_raw[(rr_raw > 200) & (rr_raw < 3000)]
diffs = np.diff(valid_rr)
sess_rmssd = np.sqrt(np.mean(diffs**2)) if len(diffs) > 0 else 0
sess_hr = 60000.0 / np.mean(valid_rr) if len(valid_rr) > 0 else 0
summary = f'Avg HR: {sess_hr:.0f} bpm  |  RMSSD: {sess_rmssd:.0f} ms'
_progress(2, f"{len(ts_raw)} beats, HR {np.nanmin(hr_rolling):.0f}-{np.nanmax(hr_rolling):.0f} bpm")

# Step 3: Audio
_progress(3)
loud_t, loudness, centroids = extract_audio_features(VIDEO_PATH)
loud_db = 20.0 * np.log10(np.clip(loudness, 1e-6, None))
t_max = min(duration, hr_t[-1] if len(hr_t) else duration, loud_t[-1] if len(loud_t) else duration)
ranges = compute_data_ranges(hr_rolling, rmssd_rolling, loud_db, centroids)
_progress(3, f"{len(loud_t)} audio frames, reel = {t_max:.0f}s")

# Step 4: Axis panel
tmpdir = tempfile.mkdtemp()
_progress(4)
axis_img = os.path.join(tmpdir, 'axes.png')
render_axis_image(ranges, axis_img)

# Step 5: Data strip
_progress(5)
data_img = os.path.join(tmpdir, 'data.png')
total_data_w = render_data_image(t_max, hr_t, hr_rolling, hr_t, rmssd_rolling,
                                  loud_t, loud_db, loud_t, centroids, ranges, data_img)
_progress(5, f"{total_data_w}x{PLOT_AREA_H}px")

# Step 6: Title bar + video prep
_progress(6)
title_img = os.path.join(tmpdir, 'title.png')
render_title_bar(HORSE_NAME, MUSIC_LABEL, summary, title_img)

temp_video = os.path.join(tmpdir, 'video_top.mp4')
subprocess.run(
    ['ffmpeg', '-y', '-i', VIDEO_PATH, '-t', str(t_max),
     '-vf', f'scale=1080:-2,pad=1080:{VIDEO_HEIGHT}:0:({VIDEO_HEIGHT}-ih)/2:color=0x1a1a2e,'
            f'setsar=1,fps={FPS}',
     '-an', '-c:v', 'libx264', '-preset', 'fast', '-crf', '23', temp_video],
    capture_output=True, timeout=1200
)
_progress(6, "video prepared")

# Step 7: Scrolling graph with cursor
_progress(7, "this is the slow part...")

cursor_frac = 0.3
cursor_offset = int(DATA_W * cursor_frac)
max_scroll = max(0, total_data_w - DATA_W)
dp = f'(t/{t_max:.6f})*{total_data_w}'
scroll_x = f'min(max(0,{dp}-{cursor_offset}),{max_scroll})'
cursor_vp = f'({dp}-min(max(0,{dp}-{cursor_offset}),{max_scroll}))'
cursor_w = 5
glow_w = 18

filter_complex = (
    f'[1:v]scale={AXIS_W}:{PLOT_AREA_H}[axis];'
    f'[2:v]crop={DATA_W}:{PLOT_AREA_H}:\'{scroll_x}\':0[cropped];'
    f'color=c=0xFF6B35@0.20:s={glow_w}x{PLOT_AREA_H}:d={t_max}:r={FPS}[glow];'
    f'color=c=0xFF6B35:s={cursor_w}x{PLOT_AREA_H}:d={t_max}:r={FPS}[cursor];'
    f'[cropped][glow]overlay=x=\'{cursor_vp}-{glow_w//2}\':y=0:shortest=1[data_glow];'
    f'[data_glow][cursor]overlay=x=\'{cursor_vp}\':y=0:shortest=1[data];'
    f'[axis][data]hstack=inputs=2[plots];'
    f'color=c=0x1a1a2e:s={WIDTH}x{GRAPH_HEIGHT}:d={t_max}:r={FPS}[bg];'
    f'[3:v]scale={WIDTH}:{TITLE_H}[title];'
    f'[bg][title]overlay=0:0[bg_t];'
    f'[bg_t][plots]overlay=0:{TITLE_H},'
    f'fps={FPS}[graph_out]'
)

temp_graphs = os.path.join(tmpdir, 'graphs.mp4')
result = subprocess.run(
    ['ffmpeg', '-y',
     '-f', 'lavfi', '-i', f'color=c=black:s=2x2:d={t_max}:r={FPS}',
     '-loop', '1', '-i', axis_img,
     '-loop', '1', '-i', data_img,
     '-loop', '1', '-i', title_img,
     '-t', str(t_max),
     '-filter_complex', filter_complex,
     '-map', '[graph_out]',
     '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
     '-pix_fmt', 'yuv420p',
     temp_graphs],
    capture_output=True, text=True, timeout=1200
)

if not os.path.exists(temp_graphs) or os.path.getsize(temp_graphs) < 1000:
    _progress(7, "ERROR creating graph video!")
    print(f'  {result.stderr[-500:]}')
else:
    _progress(7, "graph video ready")

# Step 8: Final composition
_progress(8, "combining video + graphs + audio...")

OUTPUT_FILENAME = f'{HORSE_NAME}_{MUSIC_LABEL.replace(" ", "_")}_reel.mp4'
if IN_COLAB:
    OUTPUT_PATH = os.path.join('/content', OUTPUT_FILENAME)
else:
    OUTPUT_PATH = os.path.join(os.path.dirname(VIDEO_PATH), OUTPUT_FILENAME)

result = subprocess.run(
    ['ffmpeg', '-y',
     '-i', temp_video,
     '-i', temp_graphs,
     '-i', VIDEO_PATH,
     '-filter_complex',
     f'[0:v]scale=1080:{VIDEO_HEIGHT}[top];'
     f'[1:v]scale=1080:{GRAPH_HEIGHT}[bot];'
     f'[top][bot]vstack=inputs=2[v]',
     '-map', '[v]', '-map', '2:a',
     '-t', str(t_max),
     '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
     '-c:a', 'aac', '-b:a', '192k', '-shortest',
     OUTPUT_PATH],
    capture_output=True, text=True, timeout=1200
)

shutil.rmtree(tmpdir, ignore_errors=True)
elapsed = _time.time() - start_time

if os.path.exists(OUTPUT_PATH) and os.path.getsize(OUTPUT_PATH) > 1000:
    size_mb = os.path.getsize(OUTPUT_PATH) / 1048576
    _progress(8, f"done in {elapsed:.0f}s")
    display(HTML(
        "<div style='padding:12px;background:#1a3d1a;color:#b3ffb3;"
        "border:1px solid #44ff44;border-radius:6px;margin:8px 0'>"
        f"<b>\u2705 Reel generated!</b><br>"
        f"\u2022 File: <b>{OUTPUT_FILENAME}</b><br>"
        f"\u2022 Size: <b>{size_mb:.1f} MB</b><br>"
        f"\u2022 Duration: <b>{t_max:.0f}s</b><br>"
        f"\u2022 Time: <b>{elapsed:.0f}s</b><br><br>"
        f"Run <b>Cell 6</b> to download.</div>"
    ))
else:
    _progress(8, "ERROR")
    display(HTML(
        "<div style='padding:12px;background:#5c1a1a;color:#ffb3b3;"
        "border:1px solid #ff4444;border-radius:6px;margin:8px 0'>"
        f"\u274c <b>Final video was not created.</b><br>"
        f"<pre style='color:#ff9999;font-size:11px'>{result.stderr[-500:]}</pre></div>"
    ))


In [ ]:
# @title Cell 6: Download Your Reel

if IN_COLAB:
    print(f'Downloading {OUTPUT_FILENAME}...')
    colab_files.download(OUTPUT_PATH)
    print('Check your browser downloads!')
else:
    print(f'Your reel is saved at:\n  {OUTPUT_PATH}')